In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os , sys

current = os.getcwd()
reusable = os.path.join(current , ".." , "..")
print(reusable)
sys.path.append(reusable)
from utils.modules import *


In [0]:
%fs ls "abfss://bronze@spotifyproject.dfs.core.windows.net/DimArtist/"

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimArtist/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@spotifyproject.dfs.core.windows.net/DimArtist/*")

In [0]:
df_transformed = df_artist.withColumn("artist_name",upper(col("artist_name")))

In [0]:
obj = Transform()
df_transformed = obj.dropColumn(df_transformed,["_rescued_data"])

In [0]:
df_transformed = obj.dropDuplicates(df_transformed,["artist_id"])

In [0]:
df_transformed.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimArtist/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@spotifyproject.dfs.core.windows.net/DimArtist/data")\
            .toTable("spotify_catalog.silver.DimArtist")

In [0]:
%sql
select * from spotify_catalog.silver.DimArtist